# Chapter 30: Power Production and Energy Sources

This notebook models gas turbine power generation for an offshore platform,
calculates fuel gas consumption, performs power balance analysis, and estimates
CO₂ emissions. We also investigate the effect of ambient temperature on
available gas turbine power output.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 30.1 Gas Turbine Model

We model a simple-cycle gas turbine using NeqSim's compressor and heater
equipment to represent the thermodynamic cycle:
- **Air compressor** (atmospheric → combustion pressure)
- **Combustor** (heat addition at constant pressure)
- **Power turbine** (expansion to atmospheric pressure)

The net power output is: $W_{net} = W_{turbine} - W_{compressor}$

Fuel gas consumption is calculated from the heat input and fuel LHV.

In [2]:
from neqsim import jneqsim

# Fuel gas composition (typical offshore)
fuel_gas = jneqsim.thermo.system.SystemSrkEos(273.15 + 25.0, 30.0)
fuel_gas.addComponent("methane", 0.90)
fuel_gas.addComponent("ethane", 0.05)
fuel_gas.addComponent("propane", 0.03)
fuel_gas.addComponent("CO2", 0.01)
fuel_gas.addComponent("nitrogen", 0.01)
fuel_gas.setMixingRule("classic")

ops = jneqsim.thermodynamicoperations.ThermodynamicOperations(fuel_gas)
ops.TPflash()
fuel_gas.initProperties()

# Fuel gas properties
fuel_mw = fuel_gas.getMolarMass() * 1000  # g/mol
print(f"Fuel gas MW: {fuel_mw:.1f} g/mol")

# LHV approximation for natural gas: ~50 MJ/kg
LHV_MJ_per_kg = 50.0

# Gas turbine parameters
GT_RATED_POWER_MW = 25.0  # MW rated output
GT_EFFICIENCY = 0.35  # thermal efficiency at ISO conditions
N_TURBINES = 2  # number of gas turbines

print(f"\nGas Turbine Configuration:")
print(f"  Rated power per GT: {GT_RATED_POWER_MW} MW")
print(f"  Number of GTs: {N_TURBINES}")
print(f"  Total available: {GT_RATED_POWER_MW * N_TURBINES} MW")
print(f"  Thermal efficiency: {GT_EFFICIENCY:.0%}")

Fuel gas MW: 18.0 g/mol

Gas Turbine Configuration:
  Rated power per GT: 25.0 MW
  Number of GTs: 2
  Total available: 50.0 MW
  Thermal efficiency: 35%


## 30.2 Fuel Gas Consumption vs Power Load

We calculate the fuel gas flow rate required for different power outputs.
At part load, gas turbine efficiency decreases, increasing specific fuel
consumption.

In [3]:
# Power loads from 20% to 100% of rated
load_fractions = np.linspace(0.2, 1.0, 20)
power_loads_MW = load_fractions * GT_RATED_POWER_MW

# Part-load efficiency model: eta = eta_rated * (0.5 + 0.5 * load_fraction)
# This is a simplified representation of GT part-load performance
efficiencies = GT_EFFICIENCY * (0.5 + 0.5 * load_fractions)

# Fuel consumption: Q_fuel = P_output / eta, then mass = Q / LHV
heat_input_MW = power_loads_MW / efficiencies
fuel_rate_kg_hr = heat_input_MW * 1000 / (LHV_MJ_per_kg / 3.6)  # MW to kg/hr
specific_fuel = fuel_rate_kg_hr / power_loads_MW  # kg/hr per MW

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(power_loads_MW, fuel_rate_kg_hr, 'b-o', markersize=4, linewidth=2)
ax1.set_xlabel('GT Power Output (MW)', fontsize=12)
ax1.set_ylabel('Fuel Gas Rate (kg/hr)', fontsize=12)
ax1.set_title('Fuel Gas Consumption', fontsize=13)
ax1.grid(True, alpha=0.3)

ax1_twin = ax1.twinx()
ax1_twin.plot(power_loads_MW, efficiencies * 100, 'r--s', markersize=4, linewidth=1.5, label='Efficiency')
ax1_twin.set_ylabel('Thermal Efficiency (%)', fontsize=12, color='red')
ax1_twin.legend(loc='center right', fontsize=10)

ax2.plot(load_fractions * 100, specific_fuel, 'g-^', markersize=5, linewidth=2)
ax2.set_xlabel('Load Fraction (%)', fontsize=12)
ax2.set_ylabel('Specific Fuel (kg/hr per MW)', fontsize=12)
ax2.set_title('Specific Fuel Consumption', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.suptitle('Chapter 30: Gas Turbine Fuel Consumption', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../figures/ch30_fuel_consumption.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch30_fuel_consumption.png")

Figure saved: ../figures/ch30_fuel_consumption.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_31480\3798576386.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 30.3 Power Balance: Demand vs Capacity

We define the power consumers on the platform and create a power balance
pie chart showing the breakdown of power demand.

In [4]:
# Platform power consumers
power_consumers = {
    'Export Compressor': 12.0,      # MW
    'Gas Injection Comp': 8.0,      # MW
    'Water Injection Pump': 5.0,    # MW
    'Gas Lift Comp': 3.0,           # MW
    'Utilities & HVAC': 2.5,        # MW
    'Lighting & Safety': 1.0,       # MW
    'Oil Export Pump': 1.5,         # MW
}

total_demand = sum(power_consumers.values())
total_available = GT_RATED_POWER_MW * N_TURBINES
reserve_margin = (total_available - total_demand) / total_available * 100

print("Platform Power Balance:")
print("=" * 45)
for consumer, demand in power_consumers.items():
    pct = demand / total_demand * 100
    print(f"  {consumer:<25s} {demand:6.1f} MW  ({pct:4.1f}%)")
print("-" * 45)
print(f"  {'TOTAL DEMAND':<25s} {total_demand:6.1f} MW")
print(f"  {'TOTAL AVAILABLE':<25s} {total_available:6.1f} MW")
print(f"  {'RESERVE MARGIN':<25s} {reserve_margin:6.1f} %")

# Pie chart
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.Set3(np.linspace(0, 1, len(power_consumers)))
wedges, texts, autotexts = ax.pie(
    power_consumers.values(),
    labels=power_consumers.keys(),
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    pctdistance=0.8,
    textprops={'fontsize': 10}
)
for t in autotexts:
    t.set_fontsize(9)
    t.set_fontweight('bold')

ax.set_title(f'Chapter 30: Power Demand Breakdown\nTotal: {total_demand:.1f} MW (Reserve: {reserve_margin:.0f}%)',
             fontsize=13)

plt.tight_layout()
plt.savefig("../figures/ch30_power_demand_pie.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch30_power_demand_pie.png")

Platform Power Balance:
  Export Compressor           12.0 MW  (36.4%)
  Gas Injection Comp           8.0 MW  (24.2%)
  Water Injection Pump         5.0 MW  (15.2%)
  Gas Lift Comp                3.0 MW  ( 9.1%)
  Utilities & HVAC             2.5 MW  ( 7.6%)
  Lighting & Safety            1.0 MW  ( 3.0%)
  Oil Export Pump              1.5 MW  ( 4.5%)
---------------------------------------------
  TOTAL DEMAND                33.0 MW
  TOTAL AVAILABLE             50.0 MW
  RESERVE MARGIN              34.0 %
Figure saved: ../figures/ch30_power_demand_pie.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_31480\1737732340.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 30.4 Ambient Temperature Effect on GT Power

Gas turbine output degrades at higher ambient temperatures because the air
is less dense, reducing mass flow through the compressor. A typical derating
curve follows:

$$P_{available} = P_{rated} \times \left(1 - k_{derate} \times (T_{amb} - T_{ISO})\right)$$

where $T_{ISO} = 15°C$ and $k_{derate} \approx 0.7\%$ per °C.

In [5]:
T_ISO = 15.0  # ISO standard temperature (°C)
K_DERATE = 0.007  # 0.7% per °C above ISO

ambient_temps = np.linspace(-10, 45, 30)
gt_available = []

for t_amb in ambient_temps:
    derate_factor = 1.0 - K_DERATE * max(t_amb - T_ISO, 0.0)
    # Below ISO, slight power increase (up to 5%)
    if t_amb < T_ISO:
        boost = min(0.005 * (T_ISO - t_amb), 0.05)
        derate_factor = 1.0 + boost
    gt_available.append(GT_RATED_POWER_MW * N_TURBINES * derate_factor)

gt_available = np.array(gt_available)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ambient_temps, gt_available, 'b-o', markersize=4, linewidth=2, label='Available GT Power')
ax.axhline(y=total_demand, color='red', linestyle='--', linewidth=2,
           label=f'Platform Demand ({total_demand:.1f} MW)')
ax.fill_between(ambient_temps, total_demand, gt_available,
                where=(gt_available >= total_demand), alpha=0.15, color='green', label='Reserve margin')
ax.fill_between(ambient_temps, total_demand, gt_available,
                where=(gt_available < total_demand), alpha=0.15, color='red', label='Power deficit')

ax.axvline(x=T_ISO, color='gray', linestyle=':', linewidth=1, label=f'ISO T = {T_ISO}°C')
ax.set_xlabel('Ambient Temperature (°C)', fontsize=12)
ax.set_ylabel('Power (MW)', fontsize=12)
ax.set_title('Chapter 30: GT Available Power vs Ambient Temperature', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch30_gt_ambient_temp.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch30_gt_ambient_temp.png")

Figure saved: ../figures/ch30_gt_ambient_temp.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_31480\1253065763.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 30.5 CO₂ Emissions from Gas Turbine Operation

CO₂ emissions are calculated from fuel gas composition and consumption rate.
For methane-rich fuel gas, the stoichiometric CO₂ yield is approximately
2.75 kg CO₂ per kg of natural gas burned.

In [6]:
# CO2 emission factor (kg CO2 per kg fuel gas)
# CH4 + 2O2 -> CO2 + 2H2O  =>  16 kg CH4 -> 44 kg CO2  =>  2.75 kg CO2/kg CH4
CO2_FACTOR = 2.75  # kg CO2 per kg fuel gas (simplified for ~90% methane)

# Operating at current demand
fuel_at_demand = total_demand / GT_EFFICIENCY * 1000 / (LHV_MJ_per_kg / 3.6)  # kg/hr
co2_rate_kg_hr = fuel_at_demand * CO2_FACTOR
co2_rate_tonnes_yr = co2_rate_kg_hr * 8760 / 1000

print("CO₂ Emissions at Design Conditions:")
print(f"  Fuel gas rate:    {fuel_at_demand:.0f} kg/hr")
print(f"  CO₂ rate:         {co2_rate_kg_hr:.0f} kg/hr")
print(f"  Annual CO₂:       {co2_rate_tonnes_yr:.0f} tonnes/yr")
print(f"  Specific CO₂:     {co2_rate_kg_hr / (total_demand * 1000) * 1e6:.0f} g/kWh")

# Emissions at different load levels
load_levels_pct = np.array([40, 50, 60, 70, 80, 90, 100])
co2_emissions = []
for load_pct in load_levels_pct:
    load_frac = load_pct / 100.0
    power = total_demand * load_frac
    eta = GT_EFFICIENCY * (0.5 + 0.5 * load_frac)
    fuel = power / eta * 1000 / (LHV_MJ_per_kg / 3.6)
    co2 = fuel * CO2_FACTOR * 8760 / 1000  # tonnes/yr
    co2_emissions.append(co2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(load_levels_pct, co2_emissions, width=7, color='#FF5722', edgecolor='black',
       linewidth=0.8, alpha=0.8)
ax.set_xlabel('Platform Load (%)', fontsize=12)
ax.set_ylabel('Annual CO₂ Emissions (tonnes/yr)', fontsize=12)
ax.set_title('Chapter 30: CO₂ Emissions vs Operating Load', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

for i, (load, co2) in enumerate(zip(load_levels_pct, co2_emissions)):
    ax.text(load, co2 + max(co2_emissions) * 0.02, f'{co2/1000:.0f}k',
            ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig("../figures/ch30_co2_emissions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch30_co2_emissions.png")

CO₂ Emissions at Design Conditions:
  Fuel gas rate:    6789 kg/hr
  CO₂ rate:         18669 kg/hr
  Annual CO₂:       163537 tonnes/yr
  Specific CO₂:     565714 g/kWh
Figure saved: ../figures/ch30_co2_emissions.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_31480\2177843510.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 30.6 Summary

Key findings on power production and energy management:

1. **Fuel efficiency drops at part load** — specific fuel consumption increases
   by ~30% at 50% load compared to full load, favoring fewer turbines at higher load

2. **Compression dominates power demand** — export and injection compressors
   consume ~60% of total platform power

3. **Ambient temperature derates GT power** — at 40°C, available power drops by
   ~17% from ISO-rated, which may eliminate the reserve margin in tropical locations

4. **CO₂ emissions scale non-linearly with load** — running at part load increases
   specific emissions, creating an incentive for load consolidation

For offshore platforms, the power system is often the key constraint that
limits production rate. Integrating power balance into production optimization
ensures that throughput targets are achievable within the available power envelope.